# 📊 LinkedIn Engagement Intelligence — Exploratory Data Analysis
**Author:** Solomon Uche | AI Evaluator & Data Scientist

This notebook analyzes historical LinkedIn post performance to uncover patterns that drive engagement.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

# Style
plt.style.use('dark_background')
sns.set_palette('husl')

print('Libraries loaded ✅')

## 1. Load & Inspect Data

In [ ]:
df = pd.read_csv('../../data/raw/post_metrics.csv', parse_dates=['date_posted'])

print(f'Dataset shape: {df.shape}')
print(f'Date range: {df.date_posted.min()} → {df.date_posted.max()}')
print(f'\nColumn types:')
print(df.dtypes)
df.head(10)

## 2. Summary Statistics

In [ ]:
# Filter to published posts only
published = df[df['impressions'] > 0].copy()
print(f'Published posts: {len(published)}')
print(f'\n📈 Engagement Summary:')
print(published[['impressions','reactions','comments','shares','saves','engagement_rate']].describe().round(3))

## 3. Post Type Performance

In [ ]:
type_perf = published.groupby('post_type').agg(
    avg_impressions=('impressions','mean'),
    avg_engagement_rate=('engagement_rate','mean'),
    total_reactions=('reactions','sum'),
    post_count=('post_id','count')
).round(3).sort_values('avg_impressions', ascending=False)

print('Post Type Performance:')
print(type_perf)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
type_perf['avg_impressions'].plot(kind='barh', ax=axes[0], color='#00E5FF')
axes[0].set_title('Avg Impressions by Post Type', fontsize=13)
axes[0].set_xlabel('Average Impressions')

type_perf['avg_engagement_rate'].plot(kind='barh', ax=axes[1], color='#B2FF59')
axes[1].set_title('Avg Engagement Rate by Post Type', fontsize=13)
axes[1].set_xlabel('Engagement Rate')

plt.tight_layout()
plt.savefig('../../assets/post_type_performance.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Day of Week Analysis

In [ ]:
day_order = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']
day_perf = published.groupby('day_of_week').agg(
    avg_impressions=('impressions','mean'),
    avg_engagement=('engagement_rate','mean'),
    count=('post_id','count')
).reindex(day_order).dropna()

fig = px.bar(day_perf.reset_index(), x='day_of_week', y='avg_impressions',
             color='avg_engagement', color_continuous_scale='teal',
             title='Average Impressions by Day of Week',
             labels={'avg_impressions':'Avg Impressions','day_of_week':'Day'})
fig.show()

## 5. Topic Category Analysis

In [ ]:
topic_perf = published.groupby('topic_category').agg(
    avg_impressions=('impressions','mean'),
    avg_engagement=('engagement_rate','mean'),
    total_saves=('saves','sum')
).round(3).sort_values('avg_impressions', ascending=False)

print('Topic Category Performance:')
print(topic_perf)

fig = px.scatter(topic_perf.reset_index(),
                 x='avg_impressions', y='avg_engagement',
                 text='topic_category', size='total_saves',
                 title='Impressions vs Engagement Rate by Topic (bubble = saves)',
                 color='avg_engagement', color_continuous_scale='viridis')
fig.update_traces(textposition='top center')
fig.show()

## 6. Word Count vs Engagement

In [ ]:
fig = px.scatter(published, x='word_count', y='engagement_rate',
                 color='post_type', size='impressions',
                 hover_data=['notes'],
                 trendline='ols',
                 title='Word Count vs Engagement Rate',
                 labels={'word_count':'Word Count','engagement_rate':'Engagement Rate'})
fig.show()

correlation = published['word_count'].corr(published['engagement_rate'])
print(f'Correlation between word count and engagement rate: {correlation:.3f}')

## 7. CTA Impact Analysis

In [ ]:
cta_comparison = published.groupby('has_cta').agg(
    avg_impressions=('impressions','mean'),
    avg_comments=('comments','mean'),
    avg_engagement=('engagement_rate','mean')
).round(3)
cta_comparison.index = ['No CTA','Has CTA']
print('CTA Impact on Engagement:')
print(cta_comparison)

## 8. Key Insights Summary

In [ ]:
print('=' * 60)
print('📊 LINKEDIN ENGAGEMENT ANALYSIS — KEY FINDINGS')
print('=' * 60)

best_type = type_perf['avg_impressions'].idxmax()
best_topic = topic_perf['avg_impressions'].idxmax()
top_post = published.loc[published['impressions'].idxmax()]

print(f'\n✅ Best performing post type: {best_type}')
print(f'✅ Best performing topic: {best_topic}')
print(f'✅ Top post: "{top_post["notes"]}" ({top_post["impressions"]} impressions)')
print(f'✅ Average engagement rate: {published["engagement_rate"].mean():.3f} ({published["engagement_rate"].mean()*100:.1f}%)')
print(f'✅ Total impressions (historical): {published["impressions"].sum():,}')
print('\n📌 See findings.md for full interpretation.')